# Compression Analysis

This notebook estimates compression ratio by comparing raw image size to latent-vector size.

## Setup

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Load Latents

In [ ]:
LATENTS_PATH = Path("../runs/gan_ae_cfpd/latents.npy")

if not LATENTS_PATH.exists():
    raise FileNotFoundError(f"Latent file not found: {LATENTS_PATH}")

latents = np.load(LATENTS_PATH)
print("Latent shape:", latents.shape)
print("Latent dtype:", latents.dtype)


## Estimate Compression Ratio

In [ ]:
# Update these values based on your experiment.
num_images = latents.shape[0]
image_height = 64
image_width = 64
channels = 3

raw_image_bytes_per_image = image_height * image_width * channels * 4  # float32 assumption
latent_bytes_total = latents.nbytes
latent_bytes_per_image = latent_bytes_total / num_images

compression_ratio = raw_image_bytes_per_image / latent_bytes_per_image

summary = {
    "num_images": num_images,
    "raw_image_bytes_per_image_float32": raw_image_bytes_per_image,
    "latent_bytes_total": latent_bytes_total,
    "latent_bytes_per_image": latent_bytes_per_image,
    "estimated_compression_ratio": compression_ratio
}

summary


## Create Summary Table

In [ ]:
df = pd.DataFrame([summary])
df


## Plot Raw vs Latent Size

In [ ]:
labels = ["Raw Image per Sample", "Latent per Sample"]
values = [raw_image_bytes_per_image, latent_bytes_per_image]

plt.figure()
plt.bar(labels, values)
plt.title("Raw Image Size vs Latent Vector Size")
plt.ylabel("Bytes per Sample")
plt.xticks(rotation=15)
plt.show()


## Save Compression Summary

In [ ]:
OUTPUT_CSV = Path("../runs/compression_summary.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved compression summary to: {OUTPUT_CSV}")
